Реализация адаптивных и стохастических методов $AdaGrad, AdamW$.
Подходы к борьбе с переобучением (отложенная выборка, кросс-валидация, регуляризации L1, L2):
- **Отложенная выборка(*hold-out validation*)** - это техника, при которой исходный набор данных однократно разбивается на части (например, 80% для обучения и 20% для теста) до начала тренировки модели. Модель учится исключительно на тренировочной части, а отложенная выборка используется только для объективной оценки качества и предотвращения переобучения (например, через механизм Early Stopping). 
- **Кросс-валидация(*Cross-Validation / K-Fold*)** - это техника, при которой исходный набор данных разделяется на несколько равных частей (фолдов) для циклического обучения и проверки модели. На каждой итерации модель обучается на комбинации из $K-1$ частей данных, а оставшаяся одна часть используется в качестве валидационной выборки, что позволяет протестировать алгоритм на абсолютно всех доступных данных для максимально объективной оценки качества и надежного подбора гиперпараметров (например, коэффициента регуляризации $\lambda$).

Lasso (L1-регуляризация): Штрафует за сумму модулей весов, выполняя автоматический отбор признаков.
$$ J_{LASSO} = \frac{1}{N} \sum_{i=1}^N (y_i - \hat{y}_i)^2 + \lambda \sum_{j=1}^M |w_j| $$

Ridge (L2-регуляризация): Штрафует за сумму квадратов весов, предотвращая мультиколлинеарность.
$$ J_{RIDGE} = \frac{1}{N} \sum_{i=1}^N (y_i - \hat{y}_i)^2 + \lambda \sum_{j=1}^M w_j^2 $$
где $N$ - количество объектов в выборке, $M$ - количество признаков (веса $w$ не включают в себя смещение/bias).

## *AdaGrad* (Adaptive Gradient Algorithm)
- - Источник: Duchi, J., Hazan, E., & Singer, Y. (2011). "Adaptive Subgradient Methods for Online Learning and Stochastic Optimization" (Journal of Machine Learning Research).
- Суть: алгоритм делит текущий градиент каждого параметра на корень из суммы квадратов всех его прошлых градиентов. Это снижает скорость обучения для часто обновляемых признаков и сохраняет её высокой для редких (разреженных) признаков.

Пусть $\theta_t \in \mathbb{R}^d$ - вектор оптимизируемых параметров (весов) на шаге $t$, а $L(\theta_t)$ - целевая функция потерь. Градиент функции потерь: 
$$ g_t = \nabla_{\theta} L(\theta_t) $$

Оптимизатор вводит вектор состояния $G_t \in \mathbb{R}^d$, который аккумулирует квадраты компонент градиента. Обновление состояния и параметров происходит покоординатно с использованием оператора Адамара (покоординатного умножения $\odot$):$$ G_t = G_{1:t} = G_{t-1} + g_t \odot g_t $$$$ \theta_{t+1} = \theta_t - \frac{\eta}{\sqrt{G_t} + \epsilon} \odot g_t $$Где $\eta$ - базовый темп обучения (***learning rate***), а $\epsilon > 0$ - сглаживающий параметр (обычно $10^{-7}$ или $10^{-8}$), предотвращающий деление на ноль. Из формулы накопления $G_t$ очевидно, что для любого шага выполняется неравенство $G_t \ge G_{t-1}$, что математически доказывает монотонное убывание эффективного шага.

## *AdamW* (Adam with Decoupled Weight Decay) 
- - Источник: Loshchilov, I., & Hutter, F. (2017). "Decoupled Weight Decay Regularization" (International Conference on Learning Representations).

- Суть: алгоритм изолирует шаг уменьшения весов (Weight Decay) от шага адаптивного обновления. В отличие от стандартного Adam, где градиент L2-штрафа подмешивается в общую функцию потерь и масштабируется вместе с ошибкой, AdamW вычитает регуляризационный штраф напрямую из текущего значения веса на финальном этапе операции.
 
Алгоритм разделяет вычисление градиента по целевой функции и шаг затухания весов (***Weight Decay***). Пусть $\lambda$ - коэффициент регуляризации, а $\beta_1, \beta_2 \in [0, 1)$ - параметры экспоненциального затухания. Сначала вычисляются сглаженные оценки моментов градиента (первого $m_t$ и несмещенного второго $v_t$):
$$ g_t = \nabla_{\theta} L(\theta_t)$$
$$ m_t = \beta_1 m_{t-1} + (1 - \beta_1) g_t $$
$$v_t = \beta_2 v_{t-1} + (1 - \beta_2) g_t \odot g_t $$
Поскольку моменты инициализируются нулями, они смещены в сторону нуля на первых шагах. Для коррекции этого эффекта вычисляются несмещенные оценки:
$$ \hat{m}_t = \frac{m_t}{1 - \beta_1^t}, \quad \hat{v}_t = \frac{v_t}{1 - \beta_2^t} $$
Финальный шаг обновления параметров в AdamW инкапсулирует вычтенный шаг регуляризации, полностью независимый от адаптивного знаменателя $\hat{v}_t$:
$$ \theta_{t+1} = \theta_t - \eta \lambda \theta_t - \frac{\eta}{\sqrt{\hat{v}_t} + \epsilon} \odot \hat{m}_t $$

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import KFold, train_test_split
from sklearn.preprocessing import StandardScaler
from warnings import filterwarnings
filterwarnings('ignore')

In [ ]:
def set_seed(seed=67):
    torch.manual_seed(seed)
    np.random.seed(seed)
    
# Логарифмическая сетка параметров
lambdas = np.logspace(-3, 1, num=5)

# подготовка данных с отложенной выборкой (Hold-out validation)
data = fetch_california_housing()
X_train, X_test, y_train, y_test = train_test_split(
    data.data, data.target, test_size=0.2, random_state=67
)

In [ ]:
scaler = StandardScaler()
X_train = torch.tensor(scaler.fit_transform(X_train), dtype=torch.float32)
X_test = torch.tensor(scaler.transform(X_test), dtype=torch.float32)

y_train = torch.tensor(y_train, dtype=torch.float32).view(-1, 1)
y_test = torch.tensor(y_test, dtype=torch.float32).view(-1, 1)

In [ ]:
class LinearRegression(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.linear = nn.Linear(input_dim, 1)
    def forward(self, x):
        return self.linear(x)

In [ ]:
def plot_epoch_histories(history_dict, title):
    colors = ['blue', 'orange', 'green', 'red', 'purple']
    epochs_range = range(1, len(next(iter(history_dict.values()))['train_history']) + 1)
    
    # График Loss/Epoch (Train)
    plt.figure(figsize=(10, 6)) 
    for idx, (lmbda, data_curves) in enumerate(history_dict.items()):
        color = colors[idx % len(colors)]
        plt.plot(epochs_range, data_curves['train_history'], color=color, linewidth=1.5, 
                 label=f'λ = {lmbda:.4f}')
    plt.title(f"Train Loss History — {title}", fontsize=12, fontweight='bold')
    plt.xlabel("Epochs")
    plt.ylabel("MSE Loss")
    plt.grid(True, linestyle=':', alpha=0.6)
    plt.legend()
    plt.tight_layout()
    plt.show()
    
    # График Loss/Epoch (Train)
    plt.figure(figsize=(10, 6)) 
    for idx, (lmbda, data_curves) in enumerate(history_dict.items()):
        color = colors[idx % len(colors)]
        plt.plot(epochs_range, data_curves['test_history'], color=color, linewidth=1.5, 
                 label=f'λ = {lmbda:.4f}')
    plt.title(f"Test Loss History (Hold-out) — {title}", fontsize=12, fontweight='bold')
    plt.xlabel("Epochs")
    plt.ylabel("MSE Loss")
    plt.grid(True, linestyle=':', alpha=0.6)
    plt.legend()
    plt.tight_layout()
    plt.show()

In [ ]:
def run_isolated_experiment(optimizer_name, reg_type, lmbda, epochs=1000):
    set_seed(67)
    model = LinearRegression(X_train.shape[1]) 
    criterion = nn.MSELoss()
    wd = lmbda if reg_type == 'L2' else 0.0
    if optimizer_name == 'AdaGrad':
        optimizer = torch.optim.Adagrad(model.parameters(), lr=0.01, weight_decay=wd)
    elif optimizer_name == 'AdamW':
        optimizer = torch.optim.AdamW(model.parameters(), lr=0.01, weight_decay=wd)
        
    train_history = []
    test_history = []
    for epoch in range(epochs):
        model.train()
        optimizer.zero_grad()
        train_loss = criterion(model(X_train), y_train)
        if reg_type == 'L1':
            l1_penalty = sum(p.abs().sum() for name, p in model.named_parameters() if 'bias' not in name)
            train_loss = train_loss + lmbda * l1_penalty
        train_loss.backward()
        optimizer.step()
        
        model.eval()
        with torch.no_grad():
            pure_train = criterion(model(X_train), y_train).item()
            pure_test = criterion(model(X_test), y_test).item()
        train_history.append(pure_train)
        test_history.append(pure_test)
        
    return {
        'lambda': lmbda,
        'final_train_loss': train_history[-1],
        'test_loss': test_history[-1],
        'train_history': train_history,
        'test_history': test_history
    }

In [ ]:
print("~~~~~~~~ТЕСТИРОВАНИЕ L1 ADAGRAD~~~~~~~~")
l1_adagrad_results = []
block_1_history = {}

for lmbda in lambdas:
    res = run_isolated_experiment(optimizer_name='AdaGrad', reg_type='L1', lmbda=lmbda)
    l1_adagrad_results.append(res)
    block_1_history[lmbda] = {'train_history': res['train_history'], 'test_history': res['test_history']}
    print(f"λ: {res['lambda']:.4f} | Train MSE: {res['final_train_loss']:.4f} | Test MSE: {res['test_loss']:.4f}")

best_l1_adagrad = min(l1_adagrad_results, key=lambda x: x['test_loss'])
print(f"Итог: Лучший λ = {best_l1_adagrad['lambda']:.4f} | Финальный Test MSE = {best_l1_adagrad['test_loss']:.4f}\n")
plot_epoch_histories(block_1_history, "L1 AdaGrad")

In [ ]:
print("~~~~~~~~ТЕСТИРОВАНИЕ L2 ADAGRAD~~~~~~~~")
l2_adagrad_results = []
block_2_history = {}

for lmbda in lambdas:
    res = run_isolated_experiment(optimizer_name='AdaGrad', reg_type='L2', lmbda=lmbda)
    l2_adagrad_results.append(res)
    block_2_history[lmbda] = {'train_history': res['train_history'], 'test_history': res['test_history']}
    print(f"λ: {res['lambda']:.4f} | Train MSE: {res['final_train_loss']:.4f} | Test MSE: {res['test_loss']:.4f}")

best_l2_adagrad = min(l2_adagrad_results, key=lambda x: x['test_loss'])
print(f"Итог: Лучший λ = {best_l2_adagrad['lambda']:.4f} | Финальный Test MSE = {best_l2_adagrad['test_loss']:.4f}\n")

plot_epoch_histories(block_2_history, "L2 AdaGrad")


In [ ]:
print("~~~~~~~~ТЕСТИРОВАНИЕ L1 ADAMW~~~~~~~~")
l1_adamw_results = []
block_3_history = {}

for lmbda in lambdas:
    res = run_isolated_experiment(optimizer_name='AdamW', reg_type='L1', lmbda=lmbda)
    l1_adamw_results.append(res)
    block_3_history[lmbda] = {'train_history': res['train_history'], 'test_history': res['test_history']}
    print(f"λ: {res['lambda']:.4f} | Train MSE: {res['final_train_loss']:.4f} | Test MSE: {res['test_loss']:.4f}")

best_l1_adamw = min(l1_adamw_results, key=lambda x: x['test_loss'])
print(f"Итог: Лучший λ = {best_l1_adamw['lambda']:.4f} | Финальный Test MSE = {best_l1_adamw['test_loss']:.4f}\n")

plot_epoch_histories(block_3_history, "L1 AdamW")


In [ ]:
print("~~~~~~~~ТЕСТИРОВАНИЕ L2 ADAMW~~~~~~~~")
l2_adamw_results = []
block_4_history = {}

for lmbda in lambdas:
    res = run_isolated_experiment(optimizer_name='AdamW', reg_type='L2', lmbda=lmbda)
    l2_adamw_results.append(res)
    block_4_history[lmbda] = {'train_history': res['train_history'], 'test_history': res['test_history']}
    print(f"λ: {res['lambda']:.4f} | Train MSE: {res['final_train_loss']:.4f} | Test MSE: {res['test_loss']:.4f}")

best_l2_adamw = min(l2_adamw_results, key=lambda x: x['test_loss'])
print(f"Итог: Лучший λ = {best_l2_adamw['lambda']:.4f} | Финальный Test MSE = {best_l2_adamw['test_loss']:.4f}\n")

plot_epoch_histories(block_4_history, "L2 AdamW")


In [ ]:
print("~~~~~~~~ИТОГОВОЕ СОПОСТАВЛЕНИЕ (условных) ПОБЕДИТЕЛЕЙ~~~~~~~~")
print(f"L1 AdaGrad Победитель (λ={best_l1_adagrad['lambda']:.4f}) | Финальный Test MSE: {best_l1_adagrad['test_loss']:.4f}")
print(f"L1 AdamW   Победитель (λ={best_l1_adamw['lambda']:.4f}) | Финальный Test MSE: {best_l1_adamw['test_loss']:.4f}")
print(f"L2 AdaGrad Победитель (λ={best_l2_adagrad['lambda']:.4f}) | Финальный Test MSE: {best_l2_adagrad['test_loss']:.4f}")
print(f"L2 AdamW   Победитель (λ={best_l2_adamw['lambda']:.4f}) | Финальный Test MSE: {best_l2_adamw['test_loss']:.4f}")